<a href="https://colab.research.google.com/github/thayusky/7DaysOfCode-ML/blob/main/Day3_Data_Splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# #7DaysOfCode - Day 3: Splitting the Data
Goal: split the data into training, validation and test sets, using stratified cross-validation, and compare it with other splitting techniques.

## 1. Loading and preparing the data (same steps as Day 2)

In [2]:
import kagglehub
import pandas as pd
import numpy as np

# Load the data
path = kagglehub.dataset_download("maharshipandya/-spotify-tracks-dataset")
df = pd.read_csv(path + "/dataset.csv")

# Clean: remove the extra column, missing values and duplicated tracks
df = df.drop(columns=["Unnamed: 0"]).dropna()
tracks = df.drop_duplicates(subset="track_id").copy()

# Remove zero-popularity songs that have another version
tracks["has_other_version"] = tracks.duplicated(subset=["track_name", "artists"], keep=False)
to_remove = (tracks["popularity"] == 0) & (tracks["has_other_version"])
tracks = tracks[~to_remove]

# Create the target: popular (1) if popularity >= 70, otherwise 0
conditions = [tracks["popularity"] >= 70, tracks["popularity"] < 70]
tracks["popular"] = np.select(conditions, [1, 0])

# Select the features (X) and the target (y)
features = [
    "duration_ms", "explicit", "danceability", "energy", "key",
    "loudness", "mode", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "time_signature"
]
X = tracks[features].copy()
X["explicit"] = X["explicit"].astype(int)
y = tracks["popular"]

print("X:", X.shape, "| y:", y.shape)

Using Colab cache for faster access to the '-spotify-tracks-dataset' dataset.
X: (84285, 14) | y: (84285,)


## 2. Train/test split

We keep 20% of the data aside as a test set, which will only be used at the very end. The split is stratified, so both sets keep the same proportion of popular songs.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing
    stratify=y,         # keep the same proportion of popular songs
    random_state=42     # make the split reproducible
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("Popular in training:", round(y_train.mean() * 100, 2), "%")
print("Popular in test:", round(y_test.mean() * 100, 2), "%")

Training set: (67428, 14)
Test set: (16857, 14)
Popular in training: 3.71 %
Popular in test: 3.71 %


**Finding:** the data was split into 67,428 songs for training (80%) and 16,857 for testing (20%). Thanks to the stratified split, both sets have exactly the same proportion of popular songs (3.71%).

### 📌 Key concepts: features, train/test split, overfitting and K-Fold

**X and y**
- **X** contains the **features**: the information the model uses to make a prediction (e.g. danceability, energy, loudness). In ecology, these would be the environmental variables.
- **y** contains the **target**: what we want to predict (popular = 1, not popular = 0). In ecology, this would be species presence or absence.
- `explicit` is converted from True/False to 1/0, because models only understand numbers.

**What is a train/test split?**
We split the data into two parts:
- **Training set (80%)**: the data the model learns from.
- **Test set (20%)**: data the model never sees during training, used only at the very end to check how well it works on new songs.

It is like a student: they study using exercises (training set) and are then assessed with an exam containing new questions (test set).

**How the code works**
- **(`sklearn`)** is the most widely used Machine Learning library in Python.
- `train_test_split()` returns **four things at once**: `X_train`, `X_test`, `y_train` and `y_test`. That is why there are four names before the `=`.
- `stratify=y` keeps the same proportion of popular songs (about 3.7%) in both sets.
- `random_state=42` makes the split the same every time we run the code.
- `y.mean()`: since y only contains 0s and 1s, its mean is the proportion of popular songs.

**What is overfitting?**
Overfitting happens when a model memorises the training data, including its noise, instead of learning the general pattern. It performs very well on the training data but poorly on new data, like a student who memorises the answers but does not understand the subject. In ecology, it would be like a species distribution model fitting sampling bias rather than the species' real niche.

**What is K-Fold cross-validation?**
Instead of checking the model only once, the training data is divided into K parts (folds). In each round, one fold is used for validation and the others for training. With K = 5, there are five rounds, and every song is used for validation exactly once. This gives a more reliable estimate of performance and helps to detect overfitting.

**StratifiedKFold** does the same, but keeps the same proportion of popular songs in every fold, which is essential when one class is rare.

## 3. Comparing splitting techniques

We compare three ways of creating 5 folds from the training data, checking the percentage of popular songs in each validation fold:
- **KFold without shuffling:** splits the data in its original order.
- **KFold with shuffling:** shuffles the data before splitting.
- **StratifiedKFold:** shuffles and keeps the same proportion of popular songs in every fold.

In [7]:
from sklearn.model_selection import KFold, StratifiedKFold

# A function that returns the % of popular songs in each validation fold
def popular_per_fold(splitter, X, y):
    percentages = []
    for train_idx, val_idx in splitter.split(X, y):
        percentages.append(round(y.iloc[val_idx].mean() * 100, 2))
    return percentages

splitters = {
    "KFold (no shuffle)": KFold(n_splits=5),
    "KFold (shuffle)": KFold(n_splits=5, shuffle=True, random_state=42),
    "StratifiedKFold": StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
}

for name, splitter in splitters.items():
    print(name, popular_per_fold(splitter, X_train, y_train))

KFold (no shuffle) [np.float64(3.7), np.float64(3.63), np.float64(3.6), np.float64(3.73), np.float64(3.89)]
KFold (shuffle) [np.float64(3.89), np.float64(3.54), np.float64(3.84), np.float64(3.64), np.float64(3.64)]
StratifiedKFold [np.float64(3.71), np.float64(3.71), np.float64(3.71), np.float64(3.71), np.float64(3.71)]


In [5]:
# What if we used KFold without shuffling on the data in its original order?
print("KFold (no shuffle), original order:", popular_per_fold(KFold(n_splits=5), X, y))

KFold (no shuffle), original order: [np.float64(2.83), np.float64(5.56), np.float64(3.27), np.float64(4.94), np.float64(1.94)]


**Finding:** StratifiedKFold kept exactly the same proportion of popular songs (3.71%) in all five folds, whereas both KFold versions varied between about 3.5% and 3.9%. With such a rare class, small differences matter, so StratifiedKFold gives fairer and more reliable validation folds.

### 📌 Key concepts: functions, loops and dictionaries

**`def` – creating a function**
`def` (short for *define*) creates a **function**: a named block of code that can be reused. Instead of writing the same code several times, we write it once and call it whenever we need it.

    def popular_per_fold(splitter, X, y):
        ...
        return percentages

- `popular_per_fold` is the name of the function.
- `splitter`, `X` and `y` are its **inputs** (parameters): the information we give it each time we use it.
- `return` sends the **result** back.
- The code inside the function must be indented (moved to the right).
- Note: `def` is a Python keyword, whereas `df` is just the name we gave to our DataFrame.

**`for` – repeating code with a loop**
A `for` loop repeats the same code for each item in a group:

    for train_idx, val_idx in splitter.split(X, y):

In each round, `splitter.split()` gives the positions of the training songs (`train_idx`) and the validation songs (`val_idx`), and the code inside the loop runs again.

**`{}` – dictionaries**
A dictionary stores pairs of **"key → value"**:

    splitters = {
        "KFold (no shuffle)": KFold(n_splits=5),
        "StratifiedKFold": StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    }

Here, the key is the name of the technique and the value is the technique itself. `.items()` lets us loop through both at the same time.

## 4. Stratified cross-validation loop

For each of the 5 folds, we split the training data into training and validation sets, and normalise the features. The scaler is fitted on the training fold only, so no information from the validation fold leaks into training. On Day 4, we will train our models inside this loop.

In [6]:
from sklearn.preprocessing import StandardScaler

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), start=1):
    # Split into training and validation sets for this fold
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Normalise: fit on the training fold only, then apply to both
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)

    print(f"Fold {fold}: training = {len(X_tr)} songs | validation = {len(X_val)} songs | popular in validation = {y_val.mean() * 100:.2f}%")

Fold 1: training = 53942 songs | validation = 13486 songs | popular in validation = 3.71%
Fold 2: training = 53942 songs | validation = 13486 songs | popular in validation = 3.71%
Fold 3: training = 53942 songs | validation = 13486 songs | popular in validation = 3.71%
Fold 4: training = 53943 songs | validation = 13485 songs | popular in validation = 3.71%
Fold 5: training = 53943 songs | validation = 13485 songs | popular in validation = 3.71%


**Finding:** in each of the five rounds, about 53,942 songs were used for training and 13,486 for validation, always with 3.71% of popular songs. The features were normalised in every round using only the training fold, so no information leaked from the validation fold. This loop is ready for training models on Day 4.

In [8]:
# What if we used KFold without shuffling on the data in its original order?
print("KFold (no shuffle), original order:", popular_per_fold(KFold(n_splits=5), X, y))

KFold (no shuffle), original order: [np.float64(2.83), np.float64(5.56), np.float64(3.27), np.float64(4.94), np.float64(1.94)]
